# Phase 2: STT on-device benchmark analysis (Pixel 10 Pro)

Processes the data produced by `bench.py` running on the Pixel 10 Pro with
whisper.cpp. Per-inference transcriptions and latencies are in a CSV; the experiment
metadata (decoding protocol, device info, SHA-256 hashes of the audios and models)
is in the reproducibility card JSON.

This notebook computes the quality metrics with the same functions as Phase 1, so
the cloud and on-device numbers are directly comparable.

**Configurations:** 7 variants from crossing 2 models (whisper-tiny, whisper-base)
with ggml quantisations of whisper.cpp.

**Dataset:** the same 30 WAV utterances (16 kHz mono 16-bit PCM) as Phase 1, with
verifiable SHA-256 hashes in the reproducibility card.

**Output:** `fase2_summary.csv`, one row per configuration.

## 1. Setup

Mount Google Drive to reach the CSV and JSON produced by the benchmark on the Pixel,
install the analysis dependency (`jiwer` for WER and CER), and import the libraries.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q jiwer

In [ ]:
import json
import re
from pathlib import Path

import pandas as pd
from jiwer import wer as wer_fn, cer as cer_fn

# Input paths (adjust to the Drive folder where the Pixel outputs are stored)
DRIVE_BASE = Path('/content/drive/MyDrive/fase2_resultados')
CSV_PATH   = DRIVE_BASE / 'bench_fase2.csv'
JSON_PATH  = DRIVE_BASE / 'reproducibility_fase2.json'

# Output directory (Colab local filesystem)
OUT_DIR = Path('/content/fase2_results')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Configurations evaluated in Phase 2
CONFIGS = [
    'tiny-q4_0', 'tiny-q5_0', 'tiny-q8_0',
    'base-q3_k', 'base-q4_0', 'base-q5_0', 'base-q8_0',
]

assert CSV_PATH.exists(), f'CSV not found at {CSV_PATH}'
assert JSON_PATH.exists(), f'JSON not found at {JSON_PATH}'
print(f'CSV:  {CSV_PATH}')
print(f'JSON: {JSON_PATH}')
print(f'Configurations: {len(CONFIGS)}')

## 2. Dataset and ground truth

The dataset is the 30 English campus-navigation utterances, grouped in 6 categories
of 5 phrases each: `initial_location`, `destination`, `navigation_command`,
`confirmation`, `hard_domain`, `edge_case`.

The ground truth and the domain keyword list are defined exactly as in Phase 1, so
the cloud and on-device results compare like for like.

In [ ]:
PHRASES_ALL = [
    {'id': 'utterance_01', 'category': 'initial_location',   'ground_truth': 'I am in classroom 0.3 building G'},
    {'id': 'utterance_02', 'category': 'initial_location',   'ground_truth': 'I am in classroom 2.1 on the second floor'},
    {'id': 'utterance_03', 'category': 'initial_location',   'ground_truth': 'I am at the main entrance of building G'},
    {'id': 'utterance_04', 'category': 'initial_location',   'ground_truth': 'I am in the AI laboratory on the ground floor of building G'},
    {'id': 'utterance_05', 'category': 'initial_location',   'ground_truth': 'I am next to the elevator on floor 1 of building G'},
    {'id': 'utterance_06', 'category': 'destination',        'ground_truth': 'I want to go to classroom 1.4'},
    {'id': 'utterance_07', 'category': 'destination',        'ground_truth': 'Take me to the gym in building CD'},
    {'id': 'utterance_08', 'category': 'destination',        'ground_truth': 'I want to go to the cafeteria'},
    {'id': 'utterance_09', 'category': 'destination',        'ground_truth': 'Take me to the nearest restroom on this floor'},
    {'id': 'utterance_10', 'category': 'destination',        'ground_truth': 'Could you guide me to the EPS faculty offices please'},
    {'id': 'utterance_11', 'category': 'navigation_command', 'ground_truth': 'I am in a hurry please use the stairs'},
    {'id': 'utterance_12', 'category': 'navigation_command', 'ground_truth': 'Repeat the instructions please'},
    {'id': 'utterance_13', 'category': 'navigation_command', 'ground_truth': 'I am lost I need help'},
    {'id': 'utterance_14', 'category': 'navigation_command', 'ground_truth': 'Cancel navigation and start again'},
    {'id': 'utterance_15', 'category': 'navigation_command', 'ground_truth': 'Stop and tell me where I am'},
    {'id': 'utterance_16', 'category': 'confirmation',       'ground_truth': 'Yes I am ready to continue'},
    {'id': 'utterance_17', 'category': 'confirmation',       'ground_truth': 'No that is not my location'},
    {'id': 'utterance_18', 'category': 'confirmation',       'ground_truth': 'I have arrived at the elevator on floor 2'},
    {'id': 'utterance_19', 'category': 'confirmation',       'ground_truth': 'Done I have reached my destination'},
    {'id': 'utterance_20', 'category': 'confirmation',       'ground_truth': 'Yes I can see the elevator and the doors are open'},
    {'id': 'utterance_21', 'category': 'hard_domain',        'ground_truth': 'I am in classroom 1.1 and I want to go to the EPS faculty offices'},
    {'id': 'utterance_22', 'category': 'hard_domain',        'ground_truth': 'Take me from classroom 0.1 to the AI laboratory on the ground floor'},
    {'id': 'utterance_23', 'category': 'hard_domain',        'ground_truth': 'I just came out of the elevator on the first floor of building G'},
    {'id': 'utterance_24', 'category': 'hard_domain',        'ground_truth': 'I do not know where I am I need help finding my location'},
    {'id': 'utterance_25', 'category': 'hard_domain',        'ground_truth': 'I am at the directory sign in building G next to the mentoring room'},
    {'id': 'utterance_26', 'category': 'edge_case',          'ground_truth': 'How do I get from building CD to building G'},
    {'id': 'utterance_27', 'category': 'edge_case',          'ground_truth': 'Where is the secretary office in building G'},
    {'id': 'utterance_28', 'category': 'edge_case',          'ground_truth': 'I need to find classroom 0.3 quickly'},
    {'id': 'utterance_29', 'category': 'edge_case',          'ground_truth': 'Could you tell me how to reach the AI laboratory from here'},
    {'id': 'utterance_30', 'category': 'edge_case',          'ground_truth': 'My destination is classroom 1.4 in building G please'},
]

DOMAIN_KEYWORDS = [
    '0.1', '0.3', '1.1', '1.4', '2.1',
    'building g', 'building cd',
    'elevator', 'stairs', 'cafeteria', 'ai laboratory', 'restroom',
    'faculty offices', 'main entrance', 'mentoring room',
    'secretary', 'directory sign', 'gym',
    'ground floor', 'first floor', 'second floor', 'floor 1', 'floor 2',
    'cancel', 'repeat', 'hurry', 'lost', 'stop', 'done', 'destination',
    'doors are open',
]

GT_MAP = {p['id']: p for p in PHRASES_ALL}
print(f'{len(PHRASES_ALL)} ground-truth phrases across {len({p["category"] for p in PHRASES_ALL})} categories')
print(f'{len(DOMAIN_KEYWORDS)} domain keywords tracked for DKA')

## 3. Metric functions

Same functions as Phase 1. Four quality metrics:

- **WER** (Word Error Rate): fraction of mis-transcribed words over the normalized text.
- **CER** (Character Error Rate): the same at character level.
- **Sentence Accuracy**: 1.0 if the normalized transcription equals the ground truth, else 0.0.
- **DKA** (Domain Keyword Accuracy): fraction of the domain keywords present in the ground truth that also appear in the transcription. Returns `None` when the ground truth has no keyword.

Normalization keeps decimals in numbers (`0.3`, `1.4`), lowercases, and strips
incidental punctuation. `normalize_for_dka` also converts spoken numbers
(`zero point three`) to decimal form (`0.3`) so models that dictate numbers as words
are not penalised.

In [ ]:
_PUNCT_RE = re.compile(r"[,;:!?\"()\[\]]")
_TRAIL_DOT_RE = re.compile(r"(?<!\d)\.|\.(?!\d)")
_WS_RE = re.compile(r"\s+")
_SPOKEN_NUMS = {
    'zero': '0', 'one': '1', 'two': '2', 'three': '3', 'four': '4',
    'five': '5', 'six': '6', 'seven': '7', 'eight': '8', 'nine': '9',
}

def normalize_text(text):
    if not text:
        return ''
    t = str(text).lower().strip()
    t = _PUNCT_RE.sub(' ', t)
    t = _TRAIL_DOT_RE.sub('', t)
    return _WS_RE.sub(' ', t).strip()

def normalize_for_dka(text):
    t = normalize_text(text)
    pattern = re.compile(
        r'\b(' + '|'.join(_SPOKEN_NUMS) + r')\s+point\s+(' + '|'.join(_SPOKEN_NUMS) + r')\b'
    )
    return pattern.sub(
        lambda m: f"{_SPOKEN_NUMS[m.group(1)]}.{_SPOKEN_NUMS[m.group(2)]}",
        t,
    )

def safe_wer(gt, pred):
    if not gt:
        return 0.0
    if not pred:
        return 1.0
    return float(wer_fn(normalize_text(gt), normalize_text(pred)))

def safe_cer(gt, pred):
    if not gt:
        return 0.0
    if not pred:
        return 1.0
    return float(cer_fn(normalize_text(gt), normalize_text(pred)))

def compute_sentence_accuracy(pred, gt):
    return 1.0 if normalize_text(pred) == normalize_text(gt) else 0.0

def compute_dka(pred, gt, keywords):
    pred_norm = normalize_for_dka(pred)
    gt_norm   = normalize_for_dka(gt)
    gt_kws = [kw for kw in keywords if kw in gt_norm]
    if not gt_kws:
        return None
    return sum(1 for kw in gt_kws if kw in pred_norm) / len(gt_kws)

# Sanity checks
assert normalize_text('Hello, world.') == 'hello world'
assert normalize_text('Classroom 0.3.') == 'classroom 0.3'
assert compute_sentence_accuracy('Hello, world.', 'hello world') == 1.0
assert compute_dka('I am in classroom 0.3', 'I am in classroom 0.3 building G', DOMAIN_KEYWORDS) == 0.5
assert normalize_for_dka('zero point three') == '0.3'
print('Metric functions defined and validated')

## 4. Load benchmark data

Load the inference CSV (one row per `(model, run, audio)`) and the reproducibility
card JSON, which provides the on-disk sizes and SHA-256 hashes of the models and
audios.

In [ ]:
df = pd.read_csv(CSV_PATH)
df = df[df['model'].isin(CONFIGS)].copy()
df['transcription'] = df['transcription'].fillna('')
df['latency_ms']    = pd.to_numeric(df['latency_ms'], errors='coerce')

with open(JSON_PATH) as f:
    repro = json.load(f)
MODEL_SIZES_MB = {cfg: repro['models'][cfg]['size_bytes'] / (1024 * 1024) for cfg in CONFIGS}

print(f'Loaded {len(df)} inference rows over {len(CONFIGS)} configurations')
print(f'Runs per configuration: {df.groupby("model")["run"].nunique().unique()}')
print(f'Audios per configuration: {df.groupby("model")["audio"].nunique().unique()}')

## 5. Per-row metric computation

Compute WER, CER, Sentence Accuracy and DKA for every CSV row, scoring each
transcription against its ground truth.

In [ ]:
df['ground_truth']      = df['audio'].map(lambda a: GT_MAP[a]['ground_truth'])
df['category']          = df['audio'].map(lambda a: GT_MAP[a]['category'])
df['wer']               = df.apply(lambda r: safe_wer(r['ground_truth'], r['transcription']), axis=1)
df['cer']               = df.apply(lambda r: safe_cer(r['ground_truth'], r['transcription']), axis=1)
df['sentence_accuracy'] = df.apply(lambda r: compute_sentence_accuracy(r['transcription'], r['ground_truth']), axis=1)
df['dka']               = df.apply(lambda r: compute_dka(r['transcription'], r['ground_truth'], DOMAIN_KEYWORDS), axis=1)

print(f'Per-row metrics computed for {len(df)} rows')
print(f'Mean WER: {df["wer"].mean()*100:.2f}%')
print(f'Mean DKA (over phrases with keywords): {df["dka"].dropna().mean()*100:.2f}%')

## 6. Per-utterance aggregation (3 runs collapsed)

Each `(model, audio)` has 3 rows, one per run. With greedy decoding
(`temperature=0.0`) the transcriptions, and therefore the quality metrics, are
deterministic across runs; only latency varies slightly with the scheduler.

The 3 runs per `(model, audio)` are collapsed:
- Quality metrics: `first` (identical across runs).
- Latency: mean and standard deviation of the 3 measurements.

In [ ]:
per_audio = df.groupby(['model', 'audio']).agg(
    latency_ms_mean=('latency_ms', 'mean'),
    latency_ms_std=('latency_ms', 'std'),
    transcription=('transcription', 'first'),
    ground_truth=('ground_truth', 'first'),
    category=('category', 'first'),
    wer=('wer', 'first'),
    cer=('cer', 'first'),
    sentence_accuracy=('sentence_accuracy', 'first'),
    dka=('dka', 'first'),
).reset_index()

print(f'Per-utterance rows: {len(per_audio)}  (expected: {len(CONFIGS)} × 30 = {len(CONFIGS)*30})')
per_audio.head(3)

## 7. Per-model summary

Aggregate the metrics per configuration: mean WER, CER, Sentence Accuracy and DKA
over the 30 phrases, mean and std inter-utterance latency, and the on-disk size of
the ggml file.

In [ ]:
summary_rows = []
for model, sub in per_audio.groupby('model'):
    dka_vals = sub['dka'].dropna()
    summary_rows.append({
        'model': model,
        'size_mb': round(MODEL_SIZES_MB[model], 1),
        'wer_pct': round(sub['wer'].mean() * 100, 2),
        'cer_pct': round(sub['cer'].mean() * 100, 2),
        'sentence_accuracy_pct': round(sub['sentence_accuracy'].mean() * 100, 2),
        'dka_pct': round(dka_vals.mean() * 100, 2) if len(dka_vals) else None,
        'dka_n': len(dka_vals),
        'avg_latency_ms': round(sub['latency_ms_mean'].mean(), 1),
        'avg_latency_ms_std': round(sub['latency_ms_mean'].std(), 1),
    })

summary = pd.DataFrame(summary_rows).sort_values('model').reset_index(drop=True)
summary.to_csv(OUT_DIR / 'fase2_summary.csv', index=False)
print(summary.to_string(index=False))

## 8. Rankings by criterion

The evaluated configurations ordered by each of the three criteria relevant to the
deployment decision: quality, on-disk size, and latency.

In [ ]:
cols = ['model', 'size_mb', 'wer_pct', 'cer_pct',
        'sentence_accuracy_pct', 'dka_pct', 'avg_latency_ms']

print('=' * 75)
print('FULL SUMMARY')
print('=' * 75)
print(summary[cols].to_string(index=False))

print()
print('=' * 75)
print('RANKED BY QUALITY (DKA descending)')
print('=' * 75)
print(summary.sort_values(['dka_pct', 'sentence_accuracy_pct'], ascending=False)[cols].to_string(index=False))

print()
print('=' * 75)
print('RANKED BY DISK SIZE (smallest first)')
print('=' * 75)
print(summary.sort_values('size_mb')[cols].to_string(index=False))

print()
print('=' * 75)
print('RANKED BY LATENCY (fastest first)')
print('=' * 75)
print(summary.sort_values('avg_latency_ms')[cols].to_string(index=False))

## 9. Package outputs for download

Package the contents of `OUT_DIR` into a single `.zip` for download from Colab. The
file downloads automatically to the browser.

In [ ]:
import shutil
from google.colab import files

zip_path = '/content/fase2_analysis_results.zip'
shutil.make_archive(zip_path.replace('.zip', ''), 'zip', str(OUT_DIR))

print(f'Archive: {zip_path}')
print()
print('Contents:')
for f in sorted(OUT_DIR.rglob('*')):
    if f.is_file():
        print(f'  {f.relative_to(OUT_DIR)}  ({f.stat().st_size:,} bytes)')

files.download(zip_path)